# PyTorch Autograd: Automatic Differentiation

This notebook covers PyTorch's automatic differentiation engine (autograd), which is fundamental to training neural networks.

## 1. Introduction to Autograd

Autograd is PyTorch's automatic differentiation engine that computes gradients automatically. It allows us to compute derivatives without manually implementing backpropagation.

In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Basic Tensor Operations with Gradients

In [ ]:
# Create a tensor with requires_grad=True to track gradients
x = torch.tensor(2.0, requires_grad=True)
print(f"x: {x}")
print(f"requires_grad: {x.requires_grad}")
print(f"grad_fn: {x.grad_fn}")

In [ ]:
# Perform operations and build computation graph
y = x ** 2  # y = x^2
z = y * 3 + 2  # z = 3y + 2

print(f"y: {y}")
print(f"z: {z}")
print(f"z.grad_fn: {z.grad_fn}")

In [ ]:
# Compute gradients using backward()
z.backward()

print(f"\nAfter backward():")
print(f"x.grad: {x.grad}")
print(f"Expected: dz/dx = d(3x² + 2)/dx = 6x = 6*2 = 12")

## 3. Tensors with Multiple Elements

In [ ]:
# Create a vector tensor
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
z = y.sum()  # Must have scalar output for backward()

print(f"x: {x}")
print(f"y: {y}")
print(f"z (scalar): {z}")

z.backward()
print(f"\nx.grad: {x.grad}")
print(f"Expected: [2*1, 2*2, 2*3] = [2, 4, 6]")

## 4. Gradient Accumulation

In [ ]:
# Gradients accumulate by default
x = torch.tensor(2.0, requires_grad=True)

# First computation
y = x ** 2
y.backward()
print(f"First backward - x.grad: {x.grad}")

# Second computation
y = x ** 3
y.backward()
print(f"Second backward (accumulated) - x.grad: {x.grad}")
print(f"Note: gradients are accumulated, not replaced!")

In [ ]:
# To avoid accumulation, use .zero_grad()
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()
print(f"First backward - x.grad: {x.grad}")

x.grad.zero_()  # Zero out gradients

y = x ** 3
y.backward()
print(f"After zero_grad and second backward - x.grad: {x.grad}")

## 5. Detaching Tensors from Computation Graph

In [ ]:
# Use detach() to stop gradient tracking
x = torch.tensor([1.0, 2.0], requires_grad=True)

# This builds the graph
y = x ** 2

# Detach stops tracking gradients
z = y.detach()

w = z * 3
loss = w.sum()

loss.backward()

print(f"x.grad: {x.grad}")
print(f"Note: Gradients don't flow through the detached tensor!")

## 6. no_grad() Context for Inference

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Outside no_grad context
y = x ** 2
print(f"y.requires_grad: {y.requires_grad}")

# Inside no_grad context
with torch.no_grad():
    y = x ** 2
    print(f"Inside no_grad - y.requires_grad: {y.requires_grad}")
    print(f"Inside no_grad - y.grad_fn: {y.grad_fn}")

## 7. Gradient Computation in Training

In [ ]:
# Simple linear model: y = wx + b
# Training data
x_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[2.0], [4.0], [6.0]])

# Model parameters
w = torch.tensor([[2.0]], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

# Forward pass
y_pred = x_train @ w + b

# Compute loss (MSE)
loss = ((y_pred - y_train) ** 2).mean()

print(f"Predictions: {y_pred.squeeze()}")
print(f"Loss: {loss}")

# Backward pass
loss.backward()

print(f"\nGradients:")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}")

## 8. Manual Gradient Descent Update

In [ ]:
# Learning rate
learning_rate = 0.01

# Update parameters manually
with torch.no_grad():
    w.data -= learning_rate * w.grad
    b.data -= learning_rate * b.grad

# Zero gradients for next iteration
w.grad.zero_()
b.grad.zero_()

print(f"Updated w: {w}")
print(f"Updated b: {b}")

## 9. Higher-Order Derivatives

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# First derivative
y = x ** 3  # y = x^3, dy/dx = 3x^2
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]

print(f"y: {y}")
print(f"dy/dx: {dy_dx}")
print(f"Expected: 3 * 2^2 = 12")

# Second derivative
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]
print(f"\nd²y/dx²: {d2y_dx2}")
print(f"Expected: 6x = 6 * 2 = 12")

## 10. Custom Autograd Function

In [ ]:
# Define custom function with forward and backward
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        """Forward pass: compute x^2"""
        ctx.save_for_backward(x)
        return x ** 2
    
    @staticmethod
    def backward(ctx, grad_output):
        """Backward pass: compute gradient"""
        x, = ctx.saved_tensors
        return 2 * x * grad_output

# Use custom function
x = torch.tensor(3.0, requires_grad=True)
y = Square.apply(x)
y.backward()

print(f"x: {x}")
print(f"y (x^2): {y}")
print(f"x.grad (dy/dx = 2x): {x.grad}")

## 11. Computational Graph Visualization

In [ ]:
# Build a computation graph
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# Forward computation
z = w * x + b
y = z ** 2

print("Computation Graph:")
print(f"x = {x}")
print(f"w = {w}")
print(f"b = {b}")
print(f"z = w*x + b = {z}")
print(f"y = z^2 = {y}")

# Compute gradients
y.backward()

print(f"\nGradients:")
print(f"dy/dx: {x.grad}")
print(f"dy/dw: {w.grad}")
print(f"dy/db: {b.grad}")

## 12. Training Loop Example

In [ ]:
import torch.nn as nn

# Simple training loop
x_data = torch.randn(100, 1)
y_data = 2 * x_data + 1 + torch.randn(100, 1) * 0.1

# Initialize model parameters
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Training hyperparameters
learning_rate = 0.01
epochs = 3

losses = []

for epoch in range(epochs):
    # Forward pass
    y_pred = x_data * w + b
    loss = ((y_pred - y_data) ** 2).mean()
    losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    
    # Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

print(f"\nFinal parameters:")
print(f"w: {w.item():.4f} (expected: ~2)")
print(f"b: {b.item():.4f} (expected: ~1)")

## 13. Key Takeaways

- **Autograd** automatically computes gradients through the computation graph
- **requires_grad=True** enables gradient tracking for a tensor
- **backward()** computes gradients using the chain rule
- **Gradients accumulate** by default; use .zero_grad() to reset
- **detach()** stops gradient tracking for a tensor
- **no_grad()** disables gradient computation for efficiency during inference
- **Higher-order derivatives** require create_graph=True
- **Custom functions** can be implemented for specialized operations